# Ingesting the raw taxa count matrix

Turns `Data/raw_taxa_110.csv` (1.6 GB, 168,464 x 4,680) into Parquet artifacts that load in seconds instead of minutes. The logic lives in `src/io.py`; this notebook runs it once end to end and checks the results.

Outputs (all in `data/interim/`, gitignored): `taxa_full.parquet`, `taxa_nonzero.parquet`, `taxa_prev01.npz`, `taxon_table.parquet`, `sample_depth.parquet`.

In [1]:
import sys
print(sys.executable)


c:\Python313\python.exe


In [2]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import io as taxa_io

with open(ROOT / "config" / "params.yaml") as f:
    params = yaml.safe_load(f)["ingest"]
params

{'chunksize': 2000,
 'dtype': 'int32',
 'row_group_size': 5000,
 'parquet_compression': 'snappy',
 'prevalence_filter': 0.01}

## Stream the CSV into Parquet

Single pass, `chunksize=2000` rows, `dtype=int32` for all taxa columns. Row sums (depth), per-column non-zero counts and per-column totals are accumulated as we go, so no second full read of the CSV is needed later.

In [3]:
CSV_PATH = ROOT / "Data" / "raw_taxa_110.csv"
INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

t0 = time.time()
result = taxa_io.stream_ingest(
    CSV_PATH,
    INTERIM / "taxa_full.parquet",
    chunksize=params["chunksize"],
    row_group_size=params["row_group_size"],
    compression=params["parquet_compression"],
)
elapsed = time.time() - t0
print(f"ingested {result['n_rows']} rows x {len(result['taxon_cols'])} taxa in {elapsed:.1f}s")
print("checksum (sum of all counts):", result["checksum"])

ingested 168464 rows x 4680 taxa in 252.5s
checksum (sum of all counts): 12759313470


## Taxon table and per-sample depth

`taxon_table.parquet` parses the 4,680 period-separated taxonomy strings into ranks; `sample_depth.parquet` is the per-sample read depth and non-zero taxon count accumulated during the stream above.

In [4]:
taxon_df = taxa_io.write_taxon_table(result["taxon_cols"], INTERIM / "taxon_table.parquet")
taxa_io.write_sample_depth(result["depth_df"], INTERIM / "sample_depth.parquet")

print("taxon_table shape:", taxon_df.shape)
display(taxon_df.head(3))
display(result["depth_df"].describe())

taxon_table shape: (4680, 8)


,col_index,full_string,kingdom,phylum,class,order,family,genus
0,0,Bacteria.Actinomycetota.Coriobacteriia.Corioba...,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter
1,1,Bacteria.Actinomycetota.Coriobacteriia.Corioba...,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella
2,2,Bacteria.Actinomycetota.Coriobacteriia.Corioba...,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia


,depth,n_nonzero
count,1.684640e+05,168464.00000
mean,7.573911e+04,54.91917
std,1.873663e+05,46.39650
min,1.000000e+00,1.00000
25%,2.018400e+04,21.00000
50%,3.653100e+04,48.00000
75%,7.092800e+04,79.00000
max,9.754164e+06,831.00000


## Derived artifacts

`taxa_nonzero.parquet` drops columns that are zero across every sample. `taxa_prev01.npz` keeps only taxa present in ≥ 1% of samples (the config default), as a `scipy.sparse` CSR matrix with the sample ids and original taxon column indices bundled into the same file. Both are derived by reading `taxa_full.parquet` back — the CSV is read exactly once.

In [5]:
derived = taxa_io.write_derived_artifacts(
    INTERIM / "taxa_full.parquet",
    result["taxon_cols"],
    result["col_nonzero"],
    result["col_total"],
    result["n_rows"],
    INTERIM,
    prevalence_filter=params["prevalence_filter"],
)
{k: v for k, v in derived.items() if k not in ("keep_nonzero", "keep_prev")}

{'n_nonzero_cols': 4680, 'n_prev_cols': 419, 'n_dropped_all_zero': 0}

## Sanity checks

1. Row count is exactly 168,464 in every artifact
2. Sum of all counts in `taxa_full.parquet` equals the streaming checksum
3. `taxa_nonzero` row sums are identical to `taxa_full` row sums
4. The exact all-zero column count is recorded

In [6]:
assert result["n_rows"] == 168_464, f"expected 168464 rows from the stream, got {result['n_rows']}"

taxa_full_rows = pd.read_parquet(INTERIM / "taxa_full.parquet", columns=["sample"]).shape[0]
taxa_nonzero_rows = pd.read_parquet(INTERIM / "taxa_nonzero.parquet", columns=["sample"]).shape[0]
depth_rows = pd.read_parquet(INTERIM / "sample_depth.parquet", columns=["sample"]).shape[0]
taxon_rows = taxon_df.shape[0]

assert taxa_full_rows == 168_464, taxa_full_rows
assert taxa_nonzero_rows == 168_464, taxa_nonzero_rows
assert depth_rows == 168_464, depth_rows
assert taxon_rows == 4_680, taxon_rows
print("PASS — row count exactly 168,464 in every sample-level artifact (taxon_table: 4,680 taxa rows)")

PASS — row count exactly 168,464 in every sample-level artifact (taxon_table: 4,680 taxa rows)


In [7]:
checksum_from_parquet = taxa_io.parquet_checksum(INTERIM / "taxa_full.parquet", result["taxon_cols"])
assert checksum_from_parquet == result["checksum"], (checksum_from_parquet, result["checksum"])
print(f"PASS — taxa_full.parquet checksum ({checksum_from_parquet}) == streaming checksum ({result['checksum']})")

PASS — taxa_full.parquet checksum (12759313470) == streaming checksum (12759313470)


In [8]:
# Dropping all-zero columns cannot change any row's total — verify against
# the depth already computed during the stream, rather than re-reading
# taxa_full.parquet in full a second time.
nonzero_depth = taxa_io.iter_row_sums(INTERIM / "taxa_nonzero.parquet", derived["keep_nonzero"])
merged = result["depth_df"][["sample", "depth"]].merge(
    nonzero_depth, on="sample", suffixes=("_full", "_nonzero")
)
assert len(merged) == 168_464
assert (merged["depth_full"] == merged["depth_nonzero"]).all(), "row sums changed after dropping all-zero columns"
print("PASS — taxa_nonzero row sums identical to taxa_full row sums for all 168,464 samples")

PASS — taxa_nonzero row sums identical to taxa_full row sums for all 168,464 samples


In [9]:
n_taxa = len(result["taxon_cols"])
print(f"total taxa columns:              {n_taxa}")
print(f"all-zero columns dropped:        {derived['n_dropped_all_zero']}  (earlier small-sample probe estimated ~2,068)")
print(f"retained in taxa_nonzero:        {derived['n_nonzero_cols']}  (earlier small-sample probe estimated ~2,612)")
print(f"retained at >=1% prevalence:     {derived['n_prev_cols']}  (earlier small-sample probe estimated ~421)")

total taxa columns:              4680
all-zero columns dropped:        0  (earlier small-sample probe estimated ~2,068)
retained in taxa_nonzero:        4680  (earlier small-sample probe estimated ~2,612)
retained at >=1% prevalence:     419  (earlier small-sample probe estimated ~421)


## Summary

In [10]:
for f in sorted(INTERIM.glob("*")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")

sample_depth.parquet              1.9 MB
samples_harmonized.parquet        4.1 MB
taxa_full.parquet                78.1 MB
taxa_nonzero.parquet            146.4 MB
taxa_prev01.npz                  74.8 MB
taxon_table.parquet               0.2 MB
